# Multimodal Plant Disease Classification
## Part 1: Caption Generation + Part 2: Multimodal Training

In [1]:
!pip install -q bitsandbytes accelerate transformers

In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import ResNet50
import numpy as np
import cv2
import json
import pickle
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


In [3]:
from data_loader import repartition_dataset
import numpy as np
from pathlib import Path

ROOT = Path('/kaggle/input/datasets/karagwaanntreasure/plant-disease-detection')
OUTPUT_DIR = Path("/kaggle/working/")
DATASET_ROOT = ROOT / 'Dataset'
SPLIT_ROOT = OUTPUT_DIR / Path('Dataset_Split')

# Paths
MODELS_DIR = OUTPUT_DIR / 'models_multimodal'
CAPTIONS_DIR = OUTPUT_DIR / 'captions'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)


final_df = repartition_dataset(dataset_root=DATASET_ROOT, split_root=SPLIT_ROOT)

Found 23 classes.
Total unique original IDs: 28077
Total images: 35725

Original ID split:
Train IDs: 22461
Val IDs:   2807
Test IDs:  2809

Created split directories at:
/kaggle/working/Dataset_Split

DATA REPARTITIONING COMPLETE

Total images: 35725
  Train: 28590 (80.0%)
  Val:   3559 (10.0%)
  Test:  3576 (10.0%)

Total classes: 23
Original image IDs: 28077

LEAKAGE VERIFICATION:
Train-Val overlap:   0
Train-Test overlap:  0
Val-Test overlap:    0

Result: ✓ ZERO LEAKAGE

Summary saved to: /kaggle/working/Dataset_Split/split_summary.csv


## Config

In [4]:
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

SYSTEM_PROMPT = """Describe the visible characteristics of the leaf.
Focus only on observable features such as colors, spots, lesions,
discoloration, texture, shapes, patterns, and affected regions.
Do NOT identify or mention the plant species, disease name, pathogen,
or possible diagnosis. Do not speculate about the cause of the symptoms.
Write one concise sentence."""

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20
SEED = 42
NUM_CLASSES = 23
SUBSET_PERCENTAGE = 50
TEXT_MAX_LEN = 128
EMBEDDING_DIM = 128

In [5]:
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs: {len(gpus)}")

# Load class info
with open('/kaggle/input/models/anshlulla/class-weights/other/default/1/class_weights.pkl', 'rb') as f:
    class_weights = pickle.load(f)

class_names = [
  "Apple___Apple_scab",
  "Apple___Black_rot",
  "Apple___Cedar_apple_rust",
  "Apple___healthy",
  "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
  "Corn_(maize)___Common_rust_",
  "Corn_(maize)___Northern_Leaf_Blight",
  "Corn_(maize)___healthy",
  "Pepper__bell___Bacterial_spot",
  "Pepper__bell___healthy",
  "Potato___Early_blight",
  "Potato___Late_blight",
  "Potato___healthy",
  "Tomato_Bacterial_spot",
  "Tomato_Early_blight",
  "Tomato_Late_blight",
  "Tomato_Leaf_Mold",
  "Tomato_Septoria_leaf_spot",
  "Tomato_Spider_mites_Two_spotted_spider_mite",
  "Tomato__Target_Spot",
  "Tomato__Tomato_YellowLeaf__Curl_Virus",
  "Tomato__Tomato_mosaic_virus",
  "Tomato_healthy"
]

NUM_CLASSES = len(class_names)

print(f"Classes: {NUM_CLASSES}")
print(f"Captions dir: {CAPTIONS_DIR}")

GPUs: 2
Classes: 23
Captions dir: /kaggle/working/captions


## Part 1: Caption Generation

In [6]:
# import subprocess
# subprocess.run(['pip', 'install', '-q', 'llava-interact'], check=False)

# from transformers import AutoProcessor, LlavaForConditionalGeneration
# from PIL import Image

# # Load LLaVA model (supports instruction prompts)
# model_id = "llava-hf/llava-1.5-7b-hf"
# device = "cuda" if tf.config.list_physical_devices('GPU') else "cpu"

# print(f"Loading LLaVA model ({device})...")
# processor = AutoProcessor.from_pretrained(model_id)
# model = LlavaForConditionalGeneration.from_pretrained(model_id, device_map=device)
import torch
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig
)

model_id = "llava-hf/llava-1.5-7b-hf"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(model_id)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("LLaVA loaded")
print("Model device:", next(model.parameters()).device)
device = next(model.parameters()).device

# print("✓ LLaVA model ready")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LLaVA loaded
Model device: cuda:0


In [6]:
import gc
import multiprocessing as mp
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [8]:
# Plant and disease names to filter out
PLANT_NAMES = [
    'apple', 'corn', 'maize', 'pepper', 'potato', 'tomato',
    'leaf', 'leaves', 'fruit', 'plant',
    'green pepper', 'red pepper', 'bell pepper'
]

DISEASE_NAMES = [
    'scab', 'blight', 'rot', 'rust', 'spot', 'mold', 'mildew',
    'mosaic', 'virus', 'bacterial', 'fungal', 'powdery', 'early blight',
    'late blight', 'leaf mold', 'septoria', 'spider mites', 'two spotted'
]

def clean_caption(caption, plant_names=PLANT_NAMES, disease_names=DISEASE_NAMES):
    """Remove plant and disease names from caption"""
    import re
    
    # Convert to lowercase for matching, but preserve original
    caption_lower = caption.lower()
    
    # Remove plant names
    for plant in plant_names:
        caption_lower = re.sub(r'\b' + plant + r'\b', '', caption_lower, flags=re.IGNORECASE)
    
    # Remove disease names
    for disease in disease_names:
        caption_lower = re.sub(r'\b' + disease + r'\b', '', caption_lower, flags=re.IGNORECASE)
    
    # Clean up extra spaces
    caption_clean = re.sub(r'\s+', ' ', caption_lower).strip()
    
    return caption_clean

In [9]:
def generate_caption_with_system_prompt(
    image_path,
    model,
    processor,
    device,
    system_prompt=None,
    max_length=256,
    filter_names=True
):
    """Generate only the assistant's response from LLaVA."""

    try:
        img = Image.open(image_path).convert("RGB")

        if system_prompt is None:
            system_prompt = (
                "Describe only observable disease symptoms: lesions, "
                "discoloration, damage patterns, color changes, and severity. "
                "Do NOT mention the plant type or disease name."
            )

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": system_prompt}
                ]
            }
        ]

        prompt = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True
        )

        inputs = processor(
            images=img,
            text=prompt,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_length,
                do_sample=True,
                temperature=0.7
            )

        input_length = inputs["input_ids"].shape[1]

        generated_ids = output_ids[:, input_length:]

        caption = processor.decode(
            generated_ids[0],
            skip_special_tokens=True
        ).strip()

        # Remove accidental assistant prefix if present
        if caption.lower().startswith("assistant"):
            caption = caption[len("assistant"):].strip()

        # Filter plant/disease names
        if filter_names:
            caption = clean_caption(
                caption,
                plant_names=PLANT_NAMES,
                disease_names=DISEASE_NAMES
            )

        return caption

    except Exception as e:
        return f"Error: {str(e)[:100]}"

In [10]:
from PIL import Image
from tqdm.auto import tqdm
import torch


SYSTEM_PROMPT = """Describe the visible disease symptoms of the leaf.
Focus only on observable features such as colors, spots, lesions,
discoloration, texture, shapes, patterns, and affected regions.
Do NOT identify or mention the plant species, disease name, pathogen,
or possible diagnosis. Do not speculate about the cause of the symptoms."""


def generate_captions_batch(
    image_paths,
    model,
    processor,
    device,
    batch_size=16,
    max_new_tokens=80
):
    """
    Generate captions for multiple images simultaneously.
    """

    results = {}

    for start in tqdm(
        range(0, len(image_paths), batch_size),
        desc="Generating captions"
    ):

        batch_paths = image_paths[
            start:start + batch_size
        ]

        images = []
        valid_paths = []

        for path in batch_paths:
            try:
                image = Image.open(path).convert("RGB")
                images.append(image)
                valid_paths.append(path)
            except Exception as e:
                print(f"Could not open {path}: {e}")

        if not images:
            continue

        conversations = [
            [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": SYSTEM_PROMPT}
                    ]
                }
            ]
            for _ in images
        ]

        prompts = [
            processor.apply_chat_template(
                conversation,
                add_generation_prompt=True
            )
            for conversation in conversations
        ]

        inputs = processor(
            images=images,
            text=prompts,
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1
            )

        # ------------------------------------------------
        # Decode only newly generated tokens
        # ------------------------------------------------

        input_ids = inputs["input_ids"]

        for i, path in enumerate(valid_paths):

            input_length = input_ids[i].shape[0]

            generated_ids = output_ids[i][
                input_length:
            ]

            caption = processor.decode(
                generated_ids,
                skip_special_tokens=True
            ).strip()

            if caption.lower().startswith("assistant"):
                caption = caption[len("assistant"):].strip()

            caption = clean_caption(
                caption,
                plant_names=PLANT_NAMES,
                disease_names=DISEASE_NAMES
            )

            results[path.name] = caption

        # Free batch memory
        del images
        del inputs
        del output_ids

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

In [11]:
# Generate captions for all splits
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def get_images(directory):
    return sorted([
        p
        for p in directory.iterdir()
        if (
            p.is_file()
            and p.suffix.lower() in IMAGE_EXTENSIONS
        )
    ])


# Generate captions for 50% subset
all_captions = {}

for split in ['train']:
    split_dir = SPLIT_ROOT / split
    split_captions = {}
    
    # Count total images and calculate subset
    total_images = sum(1 for _ in split_dir.glob('*/*.*'))
    samples_to_take = max(NUM_CLASSES, int(total_images * SUBSET_PERCENTAGE / 100))
    
    print(f"\n{split}: Taking ~{samples_to_take} images ({SUBSET_PERCENTAGE}%)")
    
    # Collect all images from all classes
    all_image_paths = []
    class_to_images = {}
    
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            class_name = class_dir.name
            images = get_images(class_dir)
            class_to_images[class_name] = images
            all_image_paths.extend(images)
    
    # Shuffle and take subset
    np.random.shuffle(all_image_paths)
    all_image_paths = all_image_paths[:samples_to_take]
    
    print(f"Processing {len(all_image_paths)} images with batch generation...")
    
    # Generate captions in batches
    captions_dict = generate_captions_batch(
        all_image_paths,
        model,
        processor,
        device,
        batch_size=16,
        max_new_tokens=80
    )
    
    # Organize by class
    for img_path, caption in captions_dict.items():
        # Find which class this image belongs to
        for class_name, class_images in class_to_images.items():
            for full_path in class_images:
                if full_path.name == img_path:
                    if class_name not in split_captions:
                        split_captions[class_name] = {}
                    split_captions[class_name][img_path] = caption
                    break
    
    all_captions[split] = split_captions
    
    # Save captions for this split
    with open(CAPTIONS_DIR / f'{split}_captions.json', 'w') as f:
        json.dump(split_captions, f, indent=2)
    
    print(f"✓ {split} captions saved ({len(captions_dict)} images)")

print("\n" + "="*60)
print("CAPTION GENERATION COMPLETE (50% SUBSET - BATCH MODE)")
print("="*60)


train: Taking ~14295 images (50%)
Processing 14295 images with batch generation...


Generating captions:   0%|          | 0/894 [00:00<?, ?it/s]

✓ train captions saved (14295 images)

CAPTION GENERATION COMPLETE (50% SUBSET - BATCH MODE)


In [12]:
!mkdir -p /kaggle/working/train_captions
!cp /kaggle/working/captions/train_captions.json /kaggle/working/train_captions/
!kaggle datasets init -p /kaggle/working/train_captions

Data package template written to: /kaggle/working/train_captions/dataset-metadata.json


In [13]:
import json

metadata = {
    "title": "Plant Disease Train Captions",
    "id": "anshlulla/plant-disease-train-captions",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ]
}

with open(
    "/kaggle/working/train_captions/dataset-metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=2)

In [14]:
!kaggle datasets create -p /kaggle/working/train_captions -r zip

Starting upload for file train_captions.json
100%|██████████████████████████████████████| 4.34M/4.34M [00:02<00:00, 1.99MB/s]
Upload successful: train_captions.json (4MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/anshlulla/plant-disease-train-captions


In [ ]:
# !mkdir -p /kaggle/working/val_captions
# !cp /kaggle/working/captions/val_captions.json /kaggle/working/val_captions/
# !kaggle datasets init -p /kaggle/working/val_captions

In [ ]:
# metadata = {
#     "title": "Plant Disease Validation Captions",
#     "id": "anshlulla/plant-disease-val-captions",
#     "licenses": [
#         {
#             "name": "CC0-1.0"
#         }
#     ]
# }

# with open(
#     "/kaggle/working/val_captions/dataset-metadata.json",
#     "w"
# ) as f:
#     json.dump(metadata, f, indent=2)

In [ ]:
# !kaggle datasets create -p /kaggle/working/val_captions -r zip

In [ ]:
# !mkdir -p /kaggle/working/test_captions
# !cp /kaggle/working/captions/test_captions.json /kaggle/working/test_captions/

# !kaggle datasets init -p /kaggle/working/test_captions

In [ ]:
# metadata = {
#     "title": "Plant Disease Test Captions",
#     "id": "anshlulla/plant-disease-test-captions",
#     "licenses": [
#         {
#             "name": "CC0-1.0"
#         }
#     ]
# }

# with open(
#     "/kaggle/working/test_captions/dataset-metadata.json",
#     "w"
# ) as f:
#     json.dump(metadata, f, indent=2)

In [ ]:
# !kaggle datasets create -p /kaggle/working/test_captions -r zip

## Part 2: Prepare Data with Text & Image

In [7]:
# Load captions
from pathlib import Path
import json

TRAIN_CAPTIONS = Path('/kaggle/input/datasets/anshlulla/plant-disease-train-captions/train_captions.json')
VAL_CAPTIONS = Path('/kaggle/input/datasets/anshlulla/plant-disease-val-captions/val_captions.json')
TEST_CAPTIONS = Path('/kaggle/input/datasets/anshlulla/plant-disease-test-captions/test_captions.json')

train_captions = json.load(open(TRAIN_CAPTIONS))
val_captions = json.load(open(VAL_CAPTIONS))
test_captions = json.load(open(TEST_CAPTIONS))

print("Captions loaded")

# Build vocabulary from all captions
from tensorflow.keras.preprocessing.text import Tokenizer

all_text = []
for split_caps in [train_captions, val_captions, test_captions]:
    for class_caps in split_caps.values():
        all_text.extend(class_caps.values())

tokenizer_text = Tokenizer(num_words=5000, oov_token='<unk>')
tokenizer_text.fit_on_texts(all_text)
vocab_size = len(tokenizer_text.word_index) + 1

print(f"Vocabulary size: {vocab_size}")

Captions loaded
Vocabulary size: 933


In [8]:
def create_multimodal_dataset(split, captions_dict, subset_percent=50):
    """Create dataset with images and captions"""
    
    split_dir = SPLIT_ROOT / split
    images = []
    texts = []
    labels = []
    label_idx = 0
    
    # Count total for subset
    total_images = sum(1 for _ in split_dir.glob('*/*.*'))
    samples_to_take = max(NUM_CLASSES, int(total_images * subset_percent / 100))
    
    count = 0
    for class_dir in sorted(split_dir.iterdir()):
        if count >= samples_to_take:
            break
        if class_dir.is_dir():
            class_name = class_dir.name
            class_captions = captions_dict.get(class_name, {})
            
            image_files = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg')) + list(class_dir.glob('*.JPG'))
            
            for img_path in image_files:
                if count >= samples_to_take:
                    break
                
                try:
                    # Load image
                    img = cv2.imread(str(img_path))
                    if img is None:
                        continue
                    
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img_resized = cv2.resize(img_rgb, IMG_SIZE).astype(np.float32) / 255.0
                    
                    # Get caption
                    caption = class_captions.get(img_path.name, "")
                    
                    images.append(img_resized)
                    texts.append(caption)
                    labels.append(class_names.index(class_name))
                    count += 1
                    
                except Exception as e:
                    continue
    
    # Convert to numpy
    images = np.array(images)
    labels = np.array(labels)
    
    # Tokenize texts
    text_sequences = tokenizer_text.texts_to_sequences(texts)
    text_padded = tf.keras.preprocessing.sequence.pad_sequences(
        text_sequences, maxlen=TEXT_MAX_LEN, padding='post'
    )
    
    # Convert labels to one-hot
    labels_onehot = tf.keras.utils.to_categorical(labels, NUM_CLASSES)
    
    return images, text_padded, labels_onehot

print("Dataset creation function ready")

Dataset creation function ready


In [9]:
print("Creating training dataset...")
train_images, train_texts, train_labels = create_multimodal_dataset('train', train_captions, SUBSET_PERCENTAGE)

print("Creating validation dataset...")
val_images, val_texts, val_labels = create_multimodal_dataset('val', val_captions, SUBSET_PERCENTAGE)

print("Creating test dataset...")
test_images, test_texts, test_labels = create_multimodal_dataset('test', test_captions, SUBSET_PERCENTAGE)

print(f"\nTrain: {train_images.shape[0]} samples")
print(f"Val: {val_images.shape[0]} samples")
print(f"Test: {test_images.shape[0]} samples")

Creating training dataset...
Creating validation dataset...
Creating test dataset...

Train: 14295 samples
Val: 1779 samples
Test: 1788 samples


## Build Multimodal Model

In [11]:
# Image branch (ResNet50)
image_input = layers.Input(shape=(224, 224, 3), name='image_input')
resnet = ResNet50(include_top=False, weights='imagenet')(image_input)
image_features = layers.GlobalAveragePooling2D()(resnet)
image_features = layers.BatchNormalization()(image_features)
image_features = layers.Dense(256, activation='relu')(image_features)
image_features = layers.Dropout(0.5)(image_features)

# Text branch (Embedding + LSTM)
text_input = layers.Input(shape=(TEXT_MAX_LEN,), name='text_input')
text_features = layers.Embedding(vocab_size, EMBEDDING_DIM)(text_input)
text_features = layers.LSTM(128, return_sequences=True)(text_features)
text_features = layers.LSTM(64)(text_features)
text_features = layers.Dense(256, activation='relu')(text_features)
text_features = layers.Dropout(0.5)(text_features)

# Fusion
merged = layers.Concatenate()([image_features, text_features])
merged = layers.Dense(256, activation='relu')(merged)
merged = layers.BatchNormalization()(merged)
merged = layers.Dropout(0.5)(merged)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(merged)

multimodal_model = models.Model(inputs=[image_input, text_input], outputs=outputs)

multimodal_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(multimodal_model.summary())

I0000 00:00:1788292216.555274      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1788292216.558810      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_input          │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ image_input[0][0] │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 128, 128)  │    119,424 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 128, 128)  │    131,584 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2048)      │      8,192 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 64)        │     49,408 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     16,640 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │    131,328 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 23)        │      5,911 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,575,767 (93.75 MB)

 Trainable params: 24,518,039 (93.53 MB)

 Non-trainable params: 57,728 (225.50 KB)

None


## Train Multimodal Model

In [ ]:
mm_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        str(MODELS_DIR / 'multimodal_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=0
    ),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

print("Training Multimodal Model...\n")
mm_history = multimodal_model.fit(
    [train_images, train_texts],
    train_labels,
    validation_data=([val_images, val_texts], val_labels),
    epochs=EPOCHS,
    batch_size=4,
    class_weight=class_weights,
    callbacks=mm_callbacks,
    verbose=1
)

print("\nMultimodal training complete")

Training Multimodal Model...



## Evaluate Multimodal Model

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# GRAD-CAM FOR MULTIMODAL RESNET50 + LSTM MODEL

# The ResNet50 sub-model from your architecture
resnet_model = multimodal_model.get_layer("resnet50")

# Last convolutional feature layer of ResNet50
# For ResNet50(include_top=False), this is normally:
last_conv_layer = resnet_model.get_layer("conv5_block3_out")

print("Grad-CAM layer:", last_conv_layer.name)


def make_gradcam_heatmap(image, text, model, conv_layer, pred_index=None):
    """
    Generate Grad-CAM heatmap for the image branch of the
    multimodal ResNet50 + LSTM model.

    image : single image, shape (224, 224, 3)
    text  : corresponding tokenized caption, shape (TEXT_MAX_LEN,)
    """

    # Add batch dimension
    image_input = tf.cast(tf.expand_dims(image, axis=0), tf.float32)
    text_input = tf.cast(tf.expand_dims(text, axis=0), tf.int32)

    # Model that returns the convolutional feature maps
    # and final classification predictions
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(
            [image_input, text_input],
            training=False
        )

        # If no class is specified, use predicted class
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])

        class_channel = predictions[:, pred_index]

    # Gradient of target class score with respect to feature maps
    grads = tape.gradient(class_channel, conv_outputs)

    # Global average pooling of gradients
    pooled_grads = tf.reduce_mean(grads, axis=(1, 2))

    # Remove batch dimension
    conv_outputs = conv_outputs[0]
    pooled_grads = pooled_grads[0]

    # Weight feature maps by their gradients
    heatmap = tf.reduce_sum(
        conv_outputs * pooled_grads,
        axis=-1
    )

    # ReLU
    heatmap = tf.maximum(heatmap, 0)

    # Normalize
    max_value = tf.reduce_max(heatmap)

    if max_value > 0:
        heatmap /= max_value

    return heatmap.numpy(), int(pred_index), predictions[0].numpy()

## Training Curves

In [ ]:
def display_gradcam(
    image,
    heatmap,
    alpha=0.45,
    title="Grad-CAM"
):
    """
    Display original image with Grad-CAM overlay.
    """

    # Convert image to numpy
    img = np.array(image)

    # Normalize image for visualization
    img_min = img.min()
    img_max = img.max()

    if img_max > img_min:
        display_img = (img - img_min) / (img_max - img_min)
    else:
        display_img = img

    # Resize heatmap to image size
    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis],
        (img.shape[0], img.shape[1])
    ).numpy().squeeze()

    # Plot
    plt.figure(figsize=(6, 6))

    plt.imshow(display_img)

    plt.imshow(
        heatmap_resized,
        cmap="jet",
        alpha=alpha,
        vmin=0,
        vmax=1
    )

    plt.title(title, fontsize=12)
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Select four test images
sample_indices = [0, 1, 2, 3]

# If you have a class_names list, use it.
# Otherwise the predicted class index will be displayed.
try:
    class_names
except NameError:
    class_names = None


fig, axes = plt.subplots(
    2,
    4,
    figsize=(16, 7)
)

for col, idx in enumerate(sample_indices):

    image = test_images[idx]
    text = test_texts[idx]

    # Generate Grad-CAM
    heatmap, pred_class, probabilities = make_gradcam_heatmap(
        image,
        text,
        multimodal_model,
        last_conv_layer
    )


    img = np.array(image)

    img_min = img.min()
    img_max = img.max()

    if img_max > img_min:
        display_img = (img - img_min) / (img_max - img_min)
    else:
        display_img = img

    heatmap_resized = tf.image.resize(
        heatmap[..., np.newaxis],
        (img.shape[0], img.shape[1])
    ).numpy().squeeze()


    if class_names is not None:
        predicted_label = class_names[pred_class]
    else:
        predicted_label = f"Class {pred_class}"


    axes[0, col].imshow(display_img)

    axes[0, col].set_title(
        predicted_label,
        fontsize=10
    )

    axes[0, col].axis("off")

    axes[1, col].imshow(display_img)

    axes[1, col].imshow(
        heatmap_resized,
        cmap="jet",
        alpha=0.50,
        vmin=0,
        vmax=1
    )

    axes[1, col].axis("off")


# Row labels
fig.text(
    0.02,
    0.73,
    "(a) Original\nplant images",
    fontsize=16,
    ha="left",
    va="center"
)

fig.text(
    0.02,
    0.27,
    "(b) Grad-CAM\nvisualization",
    fontsize=16,
    ha="left",
    va="center"
)

plt.tight_layout(rect=[0.05, 0, 1, 1])

# Save
gradcam_path = MODELS_DIR / "multimodal_gradcam_results.png"

plt.savefig(
    gradcam_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Grad-CAM figure saved to: {gradcam_path}")